# Expected Returns Analytics
# 
# This notebook analyzes expected returns using the enhanced v2.0 analytics pipeline:
# - **Monte Carlo Simulation** — Probabilistic upside/downside distributions
# - **Price Target Achievement** — Probability-weighted expected returns by sector
# - **Kalman Filtered Targets** — Noise-reduced price target signals
# - **Analyst Sentiment Features** — Feature-level probability analytics
# - **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
#
# Data sources: `analytics.monte_carlo_simulation`, `analytics.price_target_achievement`,
# `analytics.kalman_filtered_price_targets`, `analytics.earnings_probability_analysis`


## 1. Setup & Environment Configuration


In [1]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")

# Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24

print("✅ Environment configured")

# ── Refactored analytics modules ─────────────────────────────────────
from finance_ml.analytics.data_utils import (
    export_to_analytics_db,
    load_identifier_columns,
)
from finance_ml.analytics.probability_analytics import (
    PriceTargetAchievementModel,
    EarningsBeatProbabilityModel,
    EPSStreakAnalyzer,
    CreditRiskProbabilityModel,
    create_earnings_probability_dashboard,
)
from finance_ml.analytics.statistical_analysis import (
    kalman_filter_price_target,
    kalman_momentum_filter,
    fit_gaussian_copula,
)
from finance_ml.analytics.optimized_ops import (
    fast_monte_carlo_simulation,
    get_optimization_status,
)
from finance_ml.analytics.visualizations import (
    create_analyst_upside_scatter,
    create_valuation_vs_growth_quadrant,
)

# --- InferenceData schema (ArviZ / xarray bridge) ---
try:
    from finance_ml.analytics.inference_schema import (
        ARVIZ_AVAILABLE,
        build_monte_carlo_inference_data,
        summarize_inference_data,
    )
except ImportError:
    ARVIZ_AVAILABLE = False

# --- Probabilistic visualizations (ArviZ-backed) ---
from finance_ml.analytics.visualizations.probability_viz import (
    create_posterior_return_forest,
    create_beat_probability_posterior,
    create_ruin_probability_diagnostic,
    create_bayesian_category_ridge,
    create_tri_model_posterior_comparison,
)


✅ Environment configured


## 2. Data Acquisition


In [2]:
%%sql
SELECT * FROM analytics.monte_carlo_simulation

,ticker,name,sector,industry,region,country,exchange,last_price,pt_median,pt_spread,expected_upside_pct,upside_std,var_5_pct,prob_positive_upside,risk_reward_ratio
0,VAIAS,Vaisala Oyj,Information Technology,Electronic Equipment Instruments and Components,Europe,FI,HLSE,44.80,51.0,7.0,17.486511,3.421878,12.934321,100.00,5.110208
1,CEM,Cementir Holding N.V.,Materials,Construction Materials,Europe,NL,BIT,16.58,17.3,8.7,9.141940,10.895955,-7.826802,78.25,0.839021
2,4543,Terumo Corporation,Health Care,Health Care Equipment and Supplies,Asia / Pacific,JP,TSE,2010.50,3150.0,1400.0,51.721253,14.210031,26.795582,100.00,3.639771
3,A298380,ABL Bio Inc.,Health Care,Biotechnology,Asia / Pacific,KR,KOSDAQ,184400.00,190000.0,194000.0,0.399462,21.515827,-36.428148,52.19,0.018566
4,4612,Nippon Paint Holdings Co. Ltd.,Materials,Chemicals,Asia / Pacific,JP,TSE,1074.50,1150.0,550.0,17.781928,11.142509,2.637253,98.87,1.595864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3870,DSFIR,DSM-Firmenich AG,Materials,Chemicals,Europe,CH,ENXTAM,62.82,91.0,51.0,48.621058,16.523467,21.952206,100.00,2.942546
3871,INRN,Interroll Holding AG,Industrials,Machinery,Europe,CH,SWX,1984.00,2500.0,691.0,33.569722,7.665779,22.958116,100.00,4.379166
3872,HEIA,Heineken N.V.,Consumer Staples,Beverages,Europe,NL,ENXTAM,78.08,86.0,52.5,22.046239,14.472258,1.343441,96.81,1.523345
3873,HEIO,Heineken Holding N.V.,Consumer Staples,Beverages,Europe,NL,ENXTAM,70.85,107.0,14.0,51.033936,4.029602,44.282418,100.00,12.664760


In [3]:
%%sql
SELECT * FROM analytics.price_target_achievement

,ticker,name,country,exchange,sector,industry,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level
0,VAIAS,Vaisala Oyj,FI,HLSE,Information Technology,Electronic Equipment Instruments and Components,0.75,13.839286,13.725490,80.000000,-0.003055,85.00,10.379464,High
1,CEM,Cementir Holding N.V.,NL,BIT,Materials,Construction Materials,0.73,4.342581,50.289017,50.000000,0.015660,72.00,3.170084,Low
2,4543,Terumo Corporation,JP,TSE,Health Care,Health Care Equipment and Supplies,0.25,56.677443,44.444444,78.571429,0.000000,85.75,14.169361,Low
3,A298380,ABL Bio Inc.,KR,KOSDAQ,Health Care,Biotechnology,0.82,3.036876,102.105263,83.333333,0.000000,85.00,2.490239,Low
4,4612,Nippon Paint Holdings Co. Ltd.,JP,TSE,Materials,Chemicals,0.68,7.026524,47.826087,36.363636,0.000000,63.75,4.778036,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4182,INRN,Interroll Holding AG,CH,SWX,Industrials,Machinery,0.43,26.008065,27.640000,57.142857,0.023260,75.00,11.183468,Medium
4183,HEIA,Heineken N.V.,NL,ENXTAM,Consumer Staples,Beverages,0.60,10.143443,61.046512,65.217391,0.033990,78.25,6.086066,Low
4184,HEIO,Heineken Holding N.V.,NL,ENXTAM,Consumer Staples,Beverages,0.23,51.023289,13.084112,66.666667,0.015600,75.00,11.735356,High
4185,LOTB,Lotus Bakeries NV,BE,ENXTBR,Consumer Staples,Food Products,0.79,9.187621,43.401240,40.000000,0.042805,72.50,7.258221,Low


In [4]:
%%sql
SELECT * FROM analytics.kalman_filtered_price_targets

,ticker,name,country,exchange,sector,industry,kalman_estimate,kalman_variance,kalman_gain,signal_strength,original_price,original_target,filtered_upside
0,VAIAS,Vaisala Oyj,FI,HLSE,Information Technology,Electronic Equipment Instruments and Components,50.436369,0.090909,0.909092,10.99999,44.80,51.0,12.581180
1,CEM,Cementir Holding N.V.,NL,BIT,Materials,Construction Materials,17.234546,0.090909,0.909092,10.99999,16.58,17.3,3.947805
2,4543,Terumo Corporation,JP,TSE,Health Care,Health Care Equipment and Supplies,3046.410033,0.090909,0.909092,10.99999,2010.50,3150.0,51.524995
3,A298380,ABL Bio Inc.,KR,KOSDAQ,Health Care,Biotechnology,189490.913719,0.090909,0.909092,10.99999,184400.00,190000.0,2.760799
4,4612,Nippon Paint Holdings Co. Ltd.,JP,TSE,Materials,Chemicals,1143.136426,0.090909,0.909092,10.99999,1074.50,1150.0,6.387755
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4181,INRN,Interroll Holding AG,CH,SWX,Industrials,Machinery,2453.091336,0.090909,0.909092,10.99999,1984.00,2500.0,23.643717
4182,HEIA,Heineken N.V.,NL,ENXTAM,Consumer Staples,Beverages,85.280007,0.090909,0.909092,10.99999,78.08,86.0,9.221320
4183,HEIO,Heineken Holding N.V.,NL,ENXTAM,Consumer Staples,Beverages,103.713666,0.090909,0.909092,10.99999,70.85,107.0,46.384850
4184,LOTB,Lotus Bakeries NV,BE,ENXTBR,Consumer Staples,Food Products,11203.637149,0.090909,0.909092,10.99999,10340.00,11290.0,8.352390


In [5]:
%%sql
SELECT * FROM public.vw_features_analyst_sentiment
WHERE next_earnings >= CURRENT_DATE - (INTERVAL '6 months') AND next_earnings <= CURRENT_DATE + (INTERVAL '6 months') AND size_class <> 'Small Cap'
ORDER BY next_earnings ASC

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,NL0013995087,CEM,Cementir Holding N.V.,Europe,NL,IT,BIT,Materials,Construction Materials,Annual,...,0.067901,0.067901,0.000000,-0.283304,-0.262149,1,1,2,0.046181,0.156250
1,FI0009900682,VAIAS,Vaisala Oyj,Europe,FI,FI,HLSE,Information Technology,Electronic Equipment Instruments and Components,Annual,...,0.000000,0.000000,0.000000,0.030183,0.000000,0,0,1,-0.077382,0.000000
2,JP3419050004,6460,Sega Sammy Holdings Inc.,Asia / Pacific,JP,JP,TSE,Consumer Discretionary,Leisure Products,Final Payment,...,0.000000,-0.142857,0.089206,-0.114122,-0.033766,0,0,1,-0.048652,0.022727
3,AEA002401015,TAQA,Abu Dhabi National Energy Company PJSC,Africa / Middle East,AE,AE,ADX,Utilities,Multi-Utilities,Quarterly,...,0.000000,0.290323,-0.182796,0.004207,-0.069355,0,1,0,0.373177,0.200000
4,JP3974450003,4681,Resorttrust Inc.,Asia / Pacific,JP,JP,TSE,Consumer Discretionary,Hotels Restaurants and Leisure,Final Payment,...,0.000000,0.000000,-0.023346,-0.253353,0.093023,0,0,0,0.053941,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4182,CH0006372897,INRN,Interroll Holding AG,Europe,CH,CH,SWX,Industrials,Machinery,Annual,...,-0.047982,-0.038462,0.009626,-0.137464,-0.032554,1,2,0,0.147153,0.192857
4183,NL0000009165,HEIA,Heineken N.V.,Europe,NL,NL,ENXTAM,Consumer Staples,Beverages,Final Payment,...,0.011765,0.011765,0.007091,0.046484,0.088123,1,1,2,-0.066753,0.043478
4184,NL0000008977,HEIO,Heineken Holding N.V.,Europe,NL,NL,ENXTAM,Consumer Staples,Beverages,Final Payment,...,0.000000,0.000000,0.000000,-0.114583,0.000000,0,0,1,-0.120677,0.000000
4185,BE0003604155,LOTB,Lotus Bakeries NV,Europe,BE,BE,ENXTBR,Consumer Staples,Food Products,Annual,...,0.198514,0.147358,0.028693,0.097081,-0.032590,1,2,4,-0.183104,0.185000


## 3. Data Overview & Quality Checks


In [6]:
# Rename the DataSpell-imported variables to convenient names
# (Adjust variable names if DataSpell assigns different ones)
try:
    mc = mc_sim.copy()
except NameError:
    print("⚠️ Run the data_input cells above first")

try:
    pt = pt_a.copy()
except NameError:
    print("⚠️ Run the price_target_achievement data_input cell first")

try:
    kal = pt_kal.copy()
except NameError:
    print("⚠️ Run the kalman_filtered_price_targets data_input cell first")

print(f"Monte Carlo Simulation:        {mc.shape[0]:,} stocks × {mc.shape[1]} cols")
print(f"Price Target Achievement:      {pt.shape[0]:,} stocks × {pt.shape[1]} cols")
print(f"Kalman Filtered Targets:       {kal.shape[0]:,} stocks × {kal.shape[1]} cols")

# Summary statistics for core return metrics
display(mc[["expected_upside_pct", "var_5_pct", "prob_positive_upside", "risk_reward_ratio"]].describe().round(2))


Monte Carlo Simulation:        3,875 stocks × 15 cols
Price Target Achievement:      4,187 stocks × 14 cols
Kalman Filtered Targets:       4,186 stocks × 13 cols


,expected_upside_pct,var_5_pct,prob_positive_upside,risk_reward_ratio
count,3875.00,3875.00,3875.00,3875.00
mean,13.95,-4.38,69.15,1.52
std,25.95,22.46,33.57,7.56
min,-73.74,-79.04,0.00,-119.16
25%,-1.57,-18.01,44.82,-0.18
50%,9.32,-6.29,81.86,0.98
75%,24.46,7.69,100.00,2.41
max,238.54,142.79,100.00,148.14


## 4. Monte Carlo Simulation Analysis


### 4.1 Expected Upside Distribution


In [7]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Expected Upside Distribution", "Probability of Positive Return"),
    vertical_spacing=0.12,
)

# Clip extreme outliers for better visualization
upside_clipped = mc["expected_upside_pct"].clip(-100, 300)

fig.add_trace(
    go.Histogram(
        x=upside_clipped,
        nbinsx=80,
        marker_color=COLORS[0],
        opacity=0.75,
        name="Expected Upside %",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(
    x=mc["expected_upside_pct"].median(),
    line_dash="dot", line_color="green",
    annotation_text=f"Median: {mc['expected_upside_pct'].median():.1f}%",
    row=1, col=1,
)

# Probability of positive return - pie chart
prob_bins = pd.cut(mc["prob_positive_upside"], bins=[0, 25, 50, 75, 100],
                   labels=["0-25%", "25-50%", "50-75%", "75-100%"])
prob_counts = prob_bins.value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=prob_counts.index.astype(str),
        y=prob_counts.values,
        marker_color=[COLORS[3], COLORS[1], COLORS[0], COLORS[2]],
        name="Stock Count",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="Monte Carlo Simulation: Return Distribution Overview",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=True,
)
fig.update_xaxes(title_text="Expected Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Probability of Positive Return", row=2, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=2, col=1)
fig.show()


### 4.2 Risk-Reward by Industry


In [8]:
# Sector-level aggregation
mc_sector = (
    mc.groupby("industry")
    .agg(
        mean_upside=("expected_upside_pct", "mean"),
        median_upside=("expected_upside_pct", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_positive=("prob_positive_upside", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=False)
)

fig = px.scatter(
    mc_sector,
    x="mean_var5",
    y="mean_upside",
    size="count",
    color="industry",
    hover_name="industry",
    hover_data={"mean_prob_positive": ":.1f", "count": True},
    title="Industry Risk-Reward: Expected Upside vs Value-at-Risk (5%)",
    labels={
        "mean_var5": "Mean VaR 5% (%)",
        "mean_upside": "Mean Expected Upside (%)",
        "count": "# Stocks",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()


### 4.3 Top Opportunities — Highest Risk-Reward Ratio (Positive Upside)


In [9]:
mc_positive = mc[mc["prob_positive_upside"] >= 75].nlargest(50, "risk_reward_ratio")

fig = px.bar(
    mc_positive,
    x="ticker",
    y="expected_upside_pct",
    color="industry",
    hover_data=["name", "prob_positive_upside", "risk_reward_ratio"],
    title="Top 50 Opportunities: Highest Risk-Reward (≥75% Prob Positive)",
    labels={"expected_upside_pct": "Expected Upside (%)", "ticker": "Ticker"},
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution by Confidence Level


In [10]:
fig = px.violin(
    pt,
    x="confidence_level",
    y="achievement_probability",
    color="confidence_level",
    box=True,
    points="outliers",
    title="Price Target Achievement Probability by Confidence Level",
    labels={
        "achievement_probability": "Achievement Probability",
        "confidence_level": "Confidence Level",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=450,
)
fig.show()


### 5.2 Probability-Weighted Expected Return by Sector


In [11]:
pt_sector = (
    pt.groupby("industry")
    .agg(
        mean_expected_return=("expected_return_prob_weighted", "mean"),
        median_expected_return=("expected_return_prob_weighted", "median"),
        mean_achievement_prob=("achievement_probability", "mean"),
        mean_conviction=("analyst_conviction", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_expected_return", ascending=True)
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=pt_sector["industry"],
        x=pt_sector["mean_expected_return"],
        orientation="h",
        marker_color=[
            COLORS[2] if v >= 0 else COLORS[3]
            for v in pt_sector["mean_expected_return"]
        ],
        text=pt_sector["mean_expected_return"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean Prob-Weighted Return: %{x:.2f}%<br>"
            "Avg Achievement Prob: %{customdata[0]:.0%}<br>"
            "Avg Conviction: %{customdata[1]:.1f}<br>"
            "Stocks: %{customdata[2]}"
        ),
        customdata=pt_sector[["mean_achievement_prob", "mean_conviction", "count"]].values,
    )
)
fig.update_layout(
    title="Probability-Weighted Expected Return by Industry",
    xaxis_title="Mean Expected Return (%)",
    template=PLOTLY_TEMPLATE,
    height=1100,
    margin=dict(l=350),
)
fig.show()


### 5.3 Conviction vs Upside Potential Scatter


In [12]:
sample_pt = pt.dropna(subset=["analyst_conviction"]).sample(min(2000, len(pt)), random_state=42)
fig = px.scatter(
    sample_pt,
    x="expected_return_prob_weighted",
    y="upside_potential",
    color="achievement_probability",
    size="analyst_conviction",
    hover_name="ticker",
    hover_data=["name", "sector", "industry", "expected_return_prob_weighted", "confidence_level"],
    title="Analyst Conviction vs Upside Potential",
    labels={
        "analyst_conviction": "Analyst Conviction (%)",
        "upside_potential": "Upside Potential (%)",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig.show()


## 5.5 Earnings Beat Probability × Price Target Alignment


In [13]:
# Cross-reference: stocks with high beat probability AND high achievement probability
beat_model = EarningsBeatProbabilityModel()
_beat_source_df = beat_results  # the loaded equities DataFrame from a prior cell
beat_results = beat_model.analyze_dataframe_enhanced(
    # analyst_sentiment has forward EPS estimates, revision momentum, and reported EPS history
    # that the three-layer fusion model requires
    _beat_source_df,
    sector_col='sector' if 'sector' in _beat_source_df.columns else 'industry',
    ticker_col='ticker'
)

if len(beat_results) > 0 and len(analyst_sentiment) > 0:
    beat_pt = beat_results[['ticker', 'posterior_beat_prob', 'confidence_score']].merge(
        pt[['ticker', 'achievement_probability', 'expected_return_prob_weighted']],
        on='ticker', how='inner',
    )
    
    fig = px.scatter(
        beat_pt,
        x='posterior_beat_prob',
        y='achievement_probability',
        color='expected_return_prob_weighted',
        hover_name='ticker',
        title='Earnings Beat Probability vs Price Target Achievement',
        labels={
            'posterior_beat_prob': 'P(Beat Next Quarter)',
            'achievement_probability': 'P(Reach Price Target)',
        },
        color_continuous_scale='RdYlGn',
        template=PLOTLY_TEMPLATE,
        height=500,
    )
    fig.show()

    # Dual-signal picks: high on both dimensions
    dual_signal = beat_pt[
        (beat_pt['posterior_beat_prob'] > 0.6) &
        (beat_pt['achievement_probability'] > 0.6)
    ].nlargest(30, 'expected_return_prob_weighted')

    print(f"🎯 Dual-signal picks (high beat + high achievement): {len(dual_signal)}")
    display(dual_signal)
else:
    print("⚠️ Insufficient data for beat probability × price target alignment")


⚠️ Insufficient data for beat probability × price target alignment


## 6. Kalman Filtered Price Target Analysis


### 6.1 Kalman Filtered vs Original Upside


In [14]:
# Pre-compute the column on the full DataFrame
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100

# Sample AFTER the column exists
kal_sample = kal.sample(min(2000, len(kal)), random_state=42).copy()

# Apply signed log1p transform for axis-aligned visualization
kal_sample["filtered_upside_log"] = np.sign(kal_sample["filtered_upside"]) * np.log1p(
    np.abs(kal_sample["filtered_upside"]))
kal_sample["raw_upside_log"] = np.sign(kal_sample["raw_upside"]) * np.log1p(np.abs(kal_sample["raw_upside"]))

fig = px.scatter(
    kal_sample,
    x="filtered_upside_log",
    y="raw_upside_log",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "kalman_estimate", "original_target", "original_price", "filtered_upside", "raw_upside"],
    title="Kalman-Filtered Upside vs Raw Analyst Upside (Log-Transformed Axes)",
    labels={
        "filtered_upside_log": "Kalman Filtered Upside — sign(x)·log₁ₚ(|x|)",
        "raw_upside_log": "Raw Analyst Upside — sign(x)·log₁ₚ(|x|)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)

# Add diagonal reference line on the log-transformed scale
log_max = max(
    kal_sample["filtered_upside_log"].abs().quantile(0.99),
    kal_sample["raw_upside_log"].abs().quantile(0.99),
)
fig.add_shape(
    type="line", x0=-log_max, y0=-log_max, x1=log_max, y1=log_max,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()


### 6.2 Signal Strength Distribution by Sector


In [15]:
fig = px.box(
    kal,
    x="industry",
    y="filtered_upside",
    color="industry",
    title="Kalman-Filtered Upside Distribution by Sector",
    labels={
        "filtered_upside": "Filtered Upside (%)",
        "industry": "",
    },
    template=PLOTLY_TEMPLATE,
    height=1000,
)
fig.update_layout(
    xaxis_tickangle=-65,
    showlegend=False,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.show()


### 6.3 Kalman Noise Reduction Effectiveness


In [16]:
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100
kal["noise_reduction"] = abs(kal["raw_upside"] - kal["filtered_upside"])

noise_by_sector = (
    kal.groupby("industry")
    .agg(
        mean_noise_reduction=("noise_reduction", "mean"),
        median_raw_upside=("raw_upside", "median"),
        median_filtered_upside=("filtered_upside", "median"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_noise_reduction", ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_raw_upside"],
    name="Raw Median Upside",
    marker_color=COLORS[1],
    opacity=0.7,
))
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_filtered_upside"],
    name="Kalman-Filtered Median Upside",
    marker_color=COLORS[0],
))
fig.update_layout(
    title="Kalman Filter Impact: Raw vs Filtered Median Upside by Sector",
    yaxis_title="Median Upside (%)",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    height=1000,
    xaxis_tickangle=-85,
)
fig.show()


### 6.4 Export Kalman-Filtered Targets


In [17]:
# Regenerate Kalman-filtered targets using the refactored module
kalman_filtered_price_targets = kalman_filter_price_target(kal)

if len(kalman_filtered_price_targets) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(kalman_filtered_price_targets),
        "kalman_filtered_price_targets"
    )
    print(f"✓ Exported {len(kalman_filtered_price_targets)} rows → analytics.kalman_filtered_price_targets")


## 7. Cross-Model Comparison


### 7.1 MC Expected Upside vs Kalman Filtered Upside


In [18]:
# Merge Monte Carlo and Kalman results
mc_kal = mc.merge(
    kal[["ticker", "country", "exchange", "filtered_upside", "kalman_estimate", "original_price", "original_target"]],
    on="ticker",
    how="inner",
)

fig = px.scatter(
    mc_kal.sample(min(2000, len(mc_kal)), random_state=42),
    x="expected_upside_pct",
    y="filtered_upside",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "original_price", "kalman_estimate", "original_target", "prob_positive_upside"],
    title="Monte Carlo vs Kalman-Filtered Expected Returns",
    labels={
        "expected_upside_pct": "MC Expected Upside (%)",
        "filtered_upside": "Kalman Filtered Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.5,
)
# Diagonal reference
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=200, y1=200,
    line=dict(color="red", dash="dash", width=2),
)
fig.show()

In [19]:

# Correlation summary
corr = mc_kal[["expected_upside_pct", "filtered_upside"]].corr().iloc[0, 1]
print(f"📊 MC ↔ Kalman correlation: {corr:.3f}")


📊 MC ↔ Kalman correlation: 0.946


### 7.2 Tri-Model Alignment: MC + Kalman + Achievement


In [20]:
# Merge all three models
tri = (
    mc[["ticker", "name", "sector", "industry", "expected_upside_pct", "prob_positive_upside"]]
    .merge(
        kal[["ticker", "filtered_upside"]],
        on="ticker",
        how="inner",
    )
    .merge(
        pt[["ticker", "expected_return_prob_weighted", "achievement_probability", "confidence_level"]],
        on="ticker",
        how="inner",
    )
)

# Agreement score: all three models agree on direction
tri["mc_bullish"] = tri["expected_upside_pct"] > 0
tri["kal_bullish"] = tri["filtered_upside"] > 0
tri["pt_bullish"] = tri["expected_return_prob_weighted"] > 0
tri["agreement_score"] = (
        tri["mc_bullish"].astype(int)
        + tri["kal_bullish"].astype(int)
        + tri["pt_bullish"].astype(int)
)
tri["signal"] = tri["agreement_score"].map(
    {0: "Strong Bearish (0/3)", 1: "Bearish (1/3)", 2: "Bullish (2/3)", 3: "Strong Bullish (3/3)"}
)

fig = px.histogram(
    tri,
    x="signal",
    color="signal",
    title="Tri-Model Signal Agreement (MC + Kalman + Achievement)",
    labels={"signal": "Model Agreement", "count": "Number of Stocks"},
    color_discrete_map={
        "Strong Bearish (0/3)": COLORS[3],
        "Bearish (1/3)": COLORS[1],
        "Bullish (2/3)": COLORS[0],
        "Strong Bullish (3/3)": COLORS[2],
    },
    category_orders={"signal": [
        "Strong Bearish (0/3)", "Bearish (1/3)",
        "Bullish (2/3)", "Strong Bullish (3/3)",
    ]},
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(showlegend=False)
fig.show()

In [21]:

print(f"\n📊 Model Agreement Summary:")
print(tri["signal"].value_counts().to_string())



📊 Model Agreement Summary:
signal
Strong Bullish (3/3)    2610
Strong Bearish (0/3)     869
Bullish (2/3)            252
Bearish (1/3)            144


### 7.3 Strong Consensus Picks — All 3 Models Bullish, High Confidence


In [22]:
strong_consensus = (
    tri[
        (tri["agreement_score"] == 3)
        & (tri["prob_positive_upside"] >= 55)
        & (tri["achievement_probability"] >= 0.6)
        ]
    .nlargest(50, "expected_upside_pct")
)

if len(strong_consensus) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_upside_pct"],
        name="MC Expected Upside",
        marker_color=COLORS[0],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["filtered_upside"],
        name="Kalman Filtered Upside",
        marker_color=COLORS[1],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_return_prob_weighted"],
        name="Prob-Weighted Return",
        marker_color=COLORS[2],
    ))
    fig.update_layout(
        title=f"Top {len(strong_consensus)} Strong Consensus Picks (All 3 Models Bullish)",
        yaxis_title="Expected Return (%)",
        barmode="group",
        template=PLOTLY_TEMPLATE,
        height=500,
        xaxis_tickangle=-45,
    )
    fig.show()

    display(
        strong_consensus[["ticker", "name", "sector", "industry", "expected_upside_pct",
                          "filtered_upside", "expected_return_prob_weighted",
                          "prob_positive_upside", "achievement_probability", "confidence_level"]]
        .reset_index(drop=True)
    )
else:
    print("No stocks meet the strong consensus criteria.")


,ticker,name,sector,industry,expected_upside_pct,filtered_upside,expected_return_prob_weighted,prob_positive_upside,achievement_probability,confidence_level
0,TATATECH,Tata Technologies Limited,Information Technology,IT Services,37.708748,8.623645,5.881321,90.12,0.62,Low
1,IRB,IRB Infrastructure Developers Limited,Industrials,Construction and Engineering,37.183341,26.806022,18.576556,100.00,0.63,Low
2,AVIO,Avio S.p.A.,Industrials,Aerospace and Defense,35.887540,14.799168,11.069767,99.88,0.68,Low
3,LNTH,Lantheus Holdings Inc.,Health Care,Health Care Equipment and Supplies,34.566421,13.915472,9.796483,100.00,0.64,Low
4,PRMB,Primo Brands Corporation,Consumer Staples,Beverages,34.509569,18.123684,13.955224,99.23,0.70,Low
5,2331,Li Ning Company Limited,Consumer Discretionary,Textiles Apparel and Luxury Goods,34.461202,2.875982,2.214504,82.79,0.70,Low
6,603993,CMOC Group Limited,Materials,Metals and Mining,34.030937,22.210294,15.880346,100.00,0.65,Low
7,CMPC,Empresas CMPC S.A.,Materials,Paper and Forest Products,33.981763,7.513255,5.454618,95.55,0.66,Low
8,OGDC,Oil and Gas Development Company Limited,Energy,Oil Gas and Consumable Fuels,33.467396,15.502121,10.572437,100.00,0.62,Low
9,MIR,Mirion Technologies Inc.,Information Technology,Electronic Equipment Instruments and Components,33.245342,26.210597,19.028876,100.00,0.66,High


### 7.4 Quad-Model Alignment: MC + Kalman + Achievement + EPS Streak


In [23]:
# Quad-model agreement (4/4): MC + Kalman + PT Achievement + Beat Probability
BEAT_BULLISH_THRESHOLD = 0.6

_has_beat_data = (
        not beat_results.empty
        and 'ticker' in beat_results.columns
        and 'posterior_beat_prob' in beat_results.columns
)

if _has_beat_data and 'ticker' in tri.columns:
    beat_slim = beat_results[['ticker', 'posterior_beat_prob']].rename(
        columns={'posterior_beat_prob': 'beat_prob'}
    )
    quad = tri.merge(beat_slim, on='ticker', how='inner')

    if quad.empty:
        print("⚠️ No overlapping tickers between tri-model and beat_results")
    else:
        quad['beat_bullish_flag'] = (quad['beat_prob'] >= BEAT_BULLISH_THRESHOLD).astype(int)
        quad['quad_agreement'] = (
                quad['mc_bullish'].astype(int)
                + quad['kalman_bullish'].astype(int)
                + quad['pt_bullish'].astype(int)
                + quad['beat_bullish_flag']
        )

        fig = px.histogram(
            quad,
            x='quad_agreement',
            nbins=5,
            title='📊 Quad-Model Agreement Distribution (MC + Kalman + PT + Beat)',
            labels={'quad_agreement': 'Models Agreeing (out of 4)'},
            color_discrete_sequence=['#2E91E5'],
            template=PLOTLY_TEMPLATE,
            height=420,
        )
        fig.update_xaxes(dtick=1)
        fig.show()

        full_consensus = (quad['quad_agreement'] == 4).sum()
        no_consensus = (quad['quad_agreement'] == 0).sum()
        print(f"📊 Full consensus (4/4): {full_consensus} stocks")
        print(f"📊 No consensus  (0/4): {no_consensus} stocks")
        print(f"📊 Total quad-model coverage: {len(quad):,} stocks")
else:
    reasons = []
    if beat_results.empty:
        reasons.append("beat_results is empty")
    elif 'ticker' not in beat_results.columns:
        reasons.append("beat_results missing 'ticker' column")
    elif 'posterior_beat_prob' not in beat_results.columns:
        reasons.append("beat_results missing 'posterior_beat_prob' column")
    print(f"⚠️ Insufficient data for quad-model agreement ({'; '.join(reasons)})")

⚠️ Insufficient data for quad-model agreement (beat_results is empty)


### 7.5 Cross-Model Dependency Structure (Gaussian Copula)


In [24]:
# Measure tail dependence between MC and Kalman return signals
if len(mc_kal) > 50:
    copula_result = fit_gaussian_copula(
        mc_kal,
        features=['expected_upside_pct', 'filtered_upside']
    )
    if copula_result:
        print(f"📊 MC ↔ Kalman Dependency Analysis:")
        print(f"   Correlation matrix:\n{copula_result.get('correlation_matrix', 'N/A')}")
        print(f"   Tail dependence: {copula_result.get('tail_dependence', 'N/A')}")


📊 MC ↔ Kalman Dependency Analysis:
   Correlation matrix:
[[1.         0.94162967]
 [0.94162967 1.        ]]
   Tail dependence: {'lower': array([[1.        , 0.77720207],
       [0.77720207, 1.        ]]), 'upper': array([[1.        , 0.84455959],
       [0.84455959, 1.        ]])}


## 8. Sector Expected Returns Heatmap


In [25]:
# Aggregate all return metrics by sector
sector_returns = (
    tri.groupby("industry")
    .agg(
        mc_mean=("expected_upside_pct", "mean"),
        mc_median=("expected_upside_pct", "median"),
        kalman_mean=("filtered_upside", "mean"),
        kalman_median=("filtered_upside", "median"),
        pt_mean=("expected_return_prob_weighted", "mean"),
        pt_median=("expected_return_prob_weighted", "median"),
        pct_bullish=("agreement_score", lambda x: (x == 3).mean() * 100),
        count=("ticker", "count"),
    )
    .reset_index()
)

heatmap_data = sector_returns.set_index("industry")[
    ["mc_mean", "mc_median", "kalman_mean", "kalman_median", "pt_mean", "pt_median", "pct_bullish"]
].rename(columns={
    "mc_mean": "MC Mean",
    "mc_median": "MC Median",
    "kalman_mean": "Kalman Mean",
    "kalman_median": "Kalman Median",
    "pt_mean": "Achiev. Mean",
    "pt_median": "Achiev. Median",
    "pct_bullish": "% All Bullish",
})

fig = px.imshow(
    heatmap_data.round(1),
    color_continuous_scale="RdYlGn",
    text_auto=True,
    aspect="auto",
    title="Industry Expected Returns Heatmap (All Models)",
    labels={"color": "Value"},
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=1250,
)
fig.show()


## 9. VaR & Tail Risk Analysis


In [26]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("VaR 5% Distribution", "VaR 5% vs Expected Upside"),
    vertical_spacing=0.12,
)

# VaR distribution
var_clipped = mc["var_5_pct"].clip(-150, 300)
fig.add_trace(
    go.Histogram(
        x=var_clipped,
        nbinsx=80,
        marker_color=COLORS[3],
        opacity=0.75,
        name="VaR 5%",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="blue", row=1, col=1)

# VaR vs Expected Upside (sampled for performance)
sample = mc.sample(min(2000, len(mc)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["var_5_pct"],
        y=sample["expected_upside_pct"],
        mode="markers",
        marker=dict(
            size=4,
            color=sample["prob_positive_upside"],
            colorscale="RdYlGn",
            colorbar=dict(title="P(+)"),
            opacity=0.5,
        ),
        text=sample["name"],
        hovertemplate="%{text}<br>VaR 5%%: %{x:.1f}%<br>Expected Upside: %{y:.1f}%<extra></extra>",
        name="Stocks",
    ),
    row=2, col=1,
)
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=300, y1=300,
    line=dict(color="gray", dash="dash", width=1),
    row=2, col=1,
)

fig.update_layout(
    title="Value-at-Risk (5%) Analysis",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=False,
)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=1)
fig.update_xaxes(title_text="VaR 5% (%)", row=2, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Expected Upside (%)", row=2, col=1)
fig.show()


In [27]:
identifier_cols = load_identifier_columns()

def _reorder_with_identifiers(result_df: pd.DataFrame) -> pd.DataFrame:
    id_cols = [c for c in identifier_cols if c in result_df.columns]
    other_cols = [c for c in result_df.columns if c not in id_cols]
    return result_df[id_cols + other_cols]


## 9.5b Portfolio Monte Carlo Simulation (Top Consensus Picks)


In [28]:
from finance_ml.ml_workflow.analytics.risk import run_monte_carlo_simulation

if len(strong_consensus) >= 4:
    # Build equal-weight portfolio from strong consensus picks
    n_picks = min(20, len(strong_consensus))
    weights = np.ones(n_picks) / n_picks
    
    # This would require daily returns data — placeholder for integration
    print(f"📊 Portfolio MC ready for {n_picks} strong consensus picks")
    print("   Requires daily returns DataFrame — integrate with market data feed")


📊 Portfolio MC ready for 20 strong consensus picks
   Requires daily returns DataFrame — integrate with market data feed


## 9.5c Inline Fast MC Simulation (Numba-accelerated)


In [29]:
# Optional: Inline fast MC simulation (Numba-accelerated)
opt_status = get_optimization_status()
print(f"🔧 Numba available: {opt_status.get('numba_available')}")

# mc_inline = fast_monte_carlo_simulation(source_df, n_simulations=10000)


🔧 Numba available: False


## 9.5 InferenceData Schema Integration

Build ArviZ-compatible InferenceData from Monte Carlo simulation results
for standardised posterior analysis, diagnostics, and NetCDF export.


In [30]:
# Build InferenceData from Monte Carlo simulation results
if ARVIZ_AVAILABLE and 'mc' in dir() and len(mc) > 0:
    try:
        idata_mc = build_monte_carlo_inference_data(
            mc, mc, n_simulations=10000,
        )
        mc_summary = summarize_inference_data(idata_mc)
        print(f"✅ InferenceData built: {mc_summary.get('groups', [])}")
        print(f"   Draws: {mc_summary.get('n_draws', 0)}, Equities: {mc_summary.get('n_equities', 0)}")
        if mc_summary.get('r_hat'):
            for var, rhat_val in mc_summary['r_hat'].items():
                print(f"   R-hat ({var}): {rhat_val:.4f}")
    except Exception as e:
        print(f"⚠️ InferenceData build failed: {e}")
else:
    print('⚠️ ArviZ not available or no MC data')


✅ InferenceData built: ['posterior_predictive', 'observed_data', 'constant_data']
   Draws: 10000, Equities: 3875


## 9.6 Probabilistic Visualizations

Generate ArviZ-backed probabilistic charts from Monte Carlo and Bayesian results.


In [31]:
# Posterior return forest from Monte Carlo results
if 'mc' in dir() and len(mc) > 0:
    fig = create_posterior_return_forest(mc, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/posterior_return_forest.html')
    print('✓ Saved posterior_return_forest.html')


✓ Saved posterior_return_forest.html


In [32]:
# Tri-model posterior comparison (requires tri-model alignment DataFrame)
tri_cols = {'name', 'expected_upside_pct', 'filtered_upside', 'expected_return_prob_weighted'}
if 'strong_consensus' in dir() and tri_cols.issubset(strong_consensus.columns):
    fig = create_tri_model_posterior_comparison(strong_consensus, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/tri_model_posterior_comparison.html')
    print('✓ Saved tri_model_posterior_comparison.html')
else:
    print('⚠️ Tri-model columns not available — skipping')


✓ Saved tri_model_posterior_comparison.html


## 10. Summary Statistics


In [33]:
summary = {
    "Monte Carlo": {
        "Stocks Analyzed": len(mc),
        "Mean Expected Upside (%)": mc["expected_upside_pct"].mean().round(2),
        "Median Expected Upside (%)": mc["expected_upside_pct"].median().round(2),
        "% Stocks with Positive Upside": (mc["expected_upside_pct"] > 0).mean() * 100,
        "Mean Prob Positive (%)": mc["prob_positive_upside"].mean().round(1),
    },
    "Price Target Achievement": {
        "Stocks Analyzed": len(pt),
        "Mean Prob-Weighted Return (%)": pt["expected_return_prob_weighted"].mean().round(2),
        "Mean Achievement Prob": pt["achievement_probability"].mean().round(3),
        "High Confidence Count": (pt["confidence_level"] == "High").sum(),
        "Mean Analyst Conviction (%)": pt["analyst_conviction"].mean().round(1),
    },
    "Kalman Filter": {
        "Stocks Analyzed": len(kal),
        "Mean Filtered Upside (%)": kal["filtered_upside"].mean().round(2),
        "Median Filtered Upside (%)": kal["filtered_upside"].median().round(2),
        "Mean Signal Strength": kal["signal_strength"].mean().round(2),
        "% Positive Filtered Upside": (kal["filtered_upside"] > 0).mean() * 100,
    },
}

summary_df = pd.DataFrame(summary).T
display(summary_df)

if len(tri) > 0:
    print(f"\n🔗 Cross-Model Coverage: {len(tri):,} stocks in all 3 models")
    print(
        f"   Strong Bullish (3/3 agree): {(tri['agreement_score'] == 3).sum():,} ({(tri['agreement_score'] == 3).mean() * 100:.1f}%)")
    print(
        f"   Strong Bearish (0/3 agree): {(tri['agreement_score'] == 0).sum():,} ({(tri['agreement_score'] == 0).mean() * 100:.1f}%)")
    print(f"   MC ↔ Kalman correlation:    {tri[['expected_upside_pct', 'filtered_upside']].corr().iloc[0, 1]:.3f}")
    print(
        f"   MC ↔ Achievement corr:      {tri[['expected_upside_pct', 'expected_return_prob_weighted']].corr().iloc[0, 1]:.3f}")

print("\n✅ Expected Returns Analytics complete")


,Stocks Analyzed,Mean Expected Upside (%),Median Expected Upside (%),% Stocks with Positive Upside,Mean Prob Positive (%),Mean Prob-Weighted Return (%),Mean Achievement Prob,High Confidence Count,Mean Analyst Conviction (%),Mean Filtered Upside (%),Median Filtered Upside (%),Mean Signal Strength,% Positive Filtered Upside
Monte Carlo,3875.0,13.95,9.32,71.070968,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Target Achievement,4187.0,NaN,NaN,NaN,NaN,3.99,0.649,505.0,61.8,NaN,NaN,NaN,NaN
Kalman Filter,4186.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,8.73,11.0,72.646918



🔗 Cross-Model Coverage: 3,875 stocks in all 3 models
   Strong Bullish (3/3 agree): 2,610 (67.4%)
   Strong Bearish (0/3 agree): 869 (22.4%)
   MC ↔ Kalman correlation:    0.946
   MC ↔ Achievement corr:      0.751

✅ Expected Returns Analytics complete


## 11. Export Results to Analytics Database


In [34]:
# Export tri-model consensus results
if len(tri) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(tri),
        "expected_returns_tri_model"
    )
    print(f"✓ Exported {len(tri)} rows → analytics.expected_returns_tri_model")

# Export strong consensus picks
if len(strong_consensus) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(strong_consensus),
        "strong_consensus_picks"
    )
    print(f"✓ Exported {len(strong_consensus)} rows → analytics.strong_consensus_picks")


✓ Exported 3875 rows → analytics.expected_returns_tri_model
✓ Exported 50 rows → analytics.strong_consensus_picks
